# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset (ordered logistic regression results for adoption predictors in Northern Kenya) using the `mlcroissant` library and referencing all entities by their `@id` fields, following the Croissant specification.

### Dataset Source
The dataset is defined by a Croissant schema available at the URL provided in the cell below.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. All downstream operations will maintain references by Croissant `@id` for record sets, fields, and columns.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Set the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata and Croissant structure
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets, each with their `@id`, along with fields/columns and their own `@id` values. Reference all by their `@id` as per best practice.

In [ ]:
# List all available record sets with their @id and title/name
record_sets = list(dataset.record_sets.values())
if not record_sets:
    print("No record sets were found in the Croissant schema.")
else:
    print("Available Record Sets:")
    for rs in record_sets:
        print(f"- @id: {rs['@id']} | name: {rs.get('name','')} | description: {rs.get('description','')}")

# List fields/columns for each record set, referenced by @id
for rs in record_sets:
    print(f"\nFields/columns for Record Set @id {rs['@id']}:")
    if 'field' in rs:
        fields = rs['field']
        for field in fields:
            field_id = field.get('@id', str(field))
            field_name = field.get('name', '')
            print(f"  - Field @id: {field_id} | name: {field_name}")
    elif 'column' in rs:
        columns = rs['column']
        for col in columns:
            col_id = col.get('@id', str(col))
            col_name = col.get('name', '')
            print(f"  - Column @id: {col_id} | name: {col_name}")
    else:
        print("  (No fields or columns defined)")

# Show a preview of records for each record set
for rs in record_sets:
    print(f"\nFirst records for record set @id {rs['@id']} (up to 2 records):")
    try:
        records = list(dataset.records(record_set=rs['@id']))
        for r in records[:2]:
            print(r)
    except Exception as e:
        print(f"  Could not load records: {e}")

## 3. Data Extraction
Load data for each record set using its unique Croissant `@id` into a pandas DataFrame. All processing will reference fields by their `@id`.

In [ ]:
# Gather all record set @id values
record_set_ids = [rs['@id'] for rs in dataset.record_sets.values()]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set @id: {record_set_id}")
        print(f"Fields in DataFrame: {list(df.columns)}\n")
    except Exception as e:
        print(f"Record set @id {record_set_id}: Could not load data ({e})")

# Choose primary record set for further analysis (first one by default)
if record_set_ids:
    primary_record_set_id = record_set_ids[0]
    print(f"\nData preview for primary record set (@id: {primary_record_set_id}):")
    display(dataframes[primary_record_set_id].head())
else:
    print("No record sets available for extraction.")

## 4. Exploratory Data Analysis (EDA)
Process numeric and categorical fields using their `@id` references. The code below filters, normalizes, and groups data using specified `@id` fields. Adjust `numeric_field_id` and `group_field_id` below to match an appropriate field as identified in the overview, referencing by `@id`.

In [ ]:
# Set field @id for EDA (replace with a valid @id from overview above)
primary_df = dataframes.get(primary_record_set_id)
print(f"Column @ids in the DataFrame: {list(primary_df.columns)}")

# Example: choose numeric and group fields by inspecting above output -- update as needed for this dataset
numeric_field_id = None
group_field_id = None

# Attempt to auto-select a numeric field for demo; or set to appropriate @id manually
for col in primary_df.columns:
    # Try to pick a field with numeric type
    if pd.api.types.is_numeric_dtype(primary_df[col]):
        numeric_field_id = col
        break

if numeric_field_id is None:
    print("No numeric field found for EDA.")
else:
    print(f"Numeric field @id selected: {numeric_field_id}")

# For grouping: try to use a non-numeric, non-null column as group field
for col in primary_df.columns:
    if not pd.api.types.is_numeric_dtype(primary_df[col]) and primary_df[col].nunique() > 1:
        group_field_id = col
        break
if group_field_id:
    print(f"Group field @id selected: {group_field_id}")

if numeric_field_id:
    # Filtering and normalization
    threshold = primary_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(primary_df[numeric_field_id]) else 0
    filtered_df = primary_df[primary_df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field_id if present
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean_value")
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Below, we visualize the data distribution of the selected numeric field and explore the relationship between the chosen numeric and group fields. Update field `@id`s as necessary to match meaningful attributes.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(primary_df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    if group_field_id and group_field_id in primary_df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=primary_df[group_field_id], y=primary_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field was found for visualization.")

## 6. Conclusion
We have loaded and explored the FAIR² dataset using the `mlcroissant` library, referencing all entities by their `@id` according to the Croissant schema.

Key steps included:
- Loading the dataset and displaying basic metadata
- Listing all available record sets and fields by their `@id`
- Extracting records for each record set using their unique identifiers
- Performing basic EDA: filtering, normalizing numeric fields, and grouping by category
- Visualizing the distribution and group-wise statistics

All analysis referenced fields, columns, and record sets using their Croissant `@id` to ensure reproducibility and schema adherence. For deeper analysis, consult the dataset's detailed documentation and field definitions embedded in the schema.
